# 86BB Gut Dynamics Simulation

Simulating multi-layer gut state dynamics for the installation. Compares three approaches:

1. **Original EMA**: Single exponential moving average (alpha=0.08)
2. **Dual-layer**: Fast-reacting layer (alpha=0.3) + slow-accumulating layer (alpha=0.02), with impulse = fast - slow
3. **Physics model**: Momentum-based with state-dependent sensitivity, drag, and velocity

Each is tested with 2,000 random food selections simulating diverse public interaction.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

PLOTLY_DARK = 'plotly_dark'
OUT_DIR = Path('../outputs')

important_species = pd.read_csv(OUT_DIR / 'important_species_list.csv')['species'].tolist()
corr_matrix = pd.read_csv(OUT_DIR / 'predict1_food_species_correlations.csv', index_col=0)
health_dir_df = pd.read_csv(OUT_DIR / 'health_direction_vector.csv', index_col=0)
food_profiles_df = pd.read_csv(OUT_DIR / 'food_species_profiles.csv', index_col=0)
weight_matrix = pd.read_csv(OUT_DIR / 'food_category_weights.csv', index_col=0)

FOOD_GROUPS = list(corr_matrix.columns)
health_direction = health_dir_df['health_weight'].reindex(important_species).fillna(0).values
N_SPECIES = len(important_species)

FOOD_CATEGORY_MAP = {}
for food_id in weight_matrix.index:
    row = weight_matrix.loc[food_id]
    groups = {g: w for g, w in row.items() if w > 0}
    if groups:
        FOOD_CATEGORY_MAP[food_id] = groups

all_food_ids = list(FOOD_CATEGORY_MAP.keys())

NEUROACTIVE_PATHWAYS = {
    'butyrate_SCFA': [s for s in [
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis', 'Roseburia_intestinalis',
        'Roseburia_faecis', 'Roseburia_sp_CAG_182', 'Roseburia_sp_CAG_309',
        'Roseburia_sp_CAG_471', 'Eubacterium_hallii', 'Eubacterium_eligens',
        'Coprococcus_eutactus', 'Coprococcus_comes', 'Coprococcus_catus',
        'Anaerostipes_hadrus', 'Agathobaculum_butyriciproducens',
        'Lachnospira_pectinoschiza', 'Eubacterium_ramulus',
        'Intestinimonas_butyriciproducens'] if s in important_species],
    'GABA': [s for s in [
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Bifidobacterium_catenulatum',
        'Bifidobacterium_pseudocatenulatum', 'Bifidobacterium_animalis',
        'Lactococcus_lactis', 'Bacteroides_fragilis'] if s in important_species],
    'tryptophan_serotonin': [s for s in [
        'Streptococcus_thermophilus', 'Streptococcus_salivarius',
        'Streptococcus_parasanguinis', 'Eubacterium_hallii',
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis',
        'Coprococcus_eutactus', 'Coprococcus_comes'] if s in important_species],
    'dopamine': ['Escherichia_coli'],
    'pro_inflammatory': [s for s in [
        'Bilophila_wadsworthia', 'Ruminococcus_gnavus', 'Ruminococcus_torques',
        'Clostridium_bolteae', 'Clostridium_bolteae_CAG_59', 'Clostridium_symbiosum',
        'Hungatella_hathewayi', 'Escherichia_coli', 'Clostridium_innocuum',
        'Flavonifractor_plautii', 'Eggerthella_lenta'] if s in important_species],
    'gut_barrier': [s for s in [
        'Akkermansia_muciniphila', 'Faecalibacterium_prausnitzii',
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Roseburia_intestinalis'] if s in important_species],
}

pathway_indices = {}
for name, species_list in NEUROACTIVE_PATHWAYS.items():
    pathway_indices[name] = [important_species.index(s) for s in species_list]

def compute_bio_scores(gut_vec):
    scores = np.zeros(8)
    scores[0] = gut_vec @ health_direction
    for i, (name, idx_list) in enumerate(pathway_indices.items()):
        if idx_list:
            scores[i+1] = gut_vec[idx_list].mean()
    diversity = np.abs(gut_vec).std()
    scores[7] = diversity
    return scores

BIO_NAMES = ['health', 'butyrate', 'GABA', 'serotonin', 'dopamine', 'inflammation', 'barrier', 'diversity']

print(f'Loaded: {N_SPECIES} species, {len(all_food_ids)} foods')
print(f'Bio-score dimensions: {len(BIO_NAMES)}')
print(f'Pathways: {list(pathway_indices.keys())}')

Loaded: 135 species, 94 foods
Bio-score dimensions: 8
Pathways: ['butyrate_SCFA', 'GABA', 'tryptophan_serotonin', 'dopamine', 'pro_inflammatory', 'gut_barrier']


---
## Generate Food Sequence

Simulate 2,000 food selections. Mix of purely random selections with occasional "streaks" where someone orders similar foods several times in a row (mimicking real installation behavior).

In [2]:
np.random.seed(42)
N_SELECTIONS = 2000

sequence = []
i = 0
while i < N_SELECTIONS:
    if np.random.random() < 0.25:
        category = np.random.choice(FOOD_GROUPS)
        foods_in_cat = [f for f, groups in FOOD_CATEGORY_MAP.items()
                        if category in groups]
        if foods_in_cat:
            streak_len = np.random.randint(3, 8)
            for _ in range(min(streak_len, N_SELECTIONS - i)):
                sequence.append(np.random.choice(foods_in_cat))
                i += 1
            continue
    sequence.append(np.random.choice(all_food_ids))
    i += 1

sequence = sequence[:N_SELECTIONS]
food_signals = np.array([food_profiles_df.loc[f].values for f in sequence])

print(f'Food sequence: {len(sequence)} selections')
print(f'Unique foods used: {len(set(sequence))}')

from collections import Counter
cat_counts = Counter()
for f in sequence:
    for g in FOOD_CATEGORY_MAP.get(f, {}):
        cat_counts[g] += 1
print(f'\nTop 5 food groups in sequence:')
for g, c in cat_counts.most_common(5):
    print(f'  {g}: {c}')

Food sequence: 2000 selections
Unique foods used: 94

Top 5 food groups in sequence:
  Vegetables: 568
  Refined_grains: 553
  Dairy: 342
  Vegetable_oils: 336
  Meat: 316


---
## Model 1: Original EMA (Baseline)

Single exponential moving average: `gut = (1-α) * gut + α * signal`, α=0.08

In [3]:
def simulate_ema(signals, alpha=0.08):
    n, d = signals.shape
    gut = np.zeros(d)
    trajectory = np.zeros((n+1, d))
    bio_trajectory = np.zeros((n+1, len(BIO_NAMES)))
    bio_trajectory[0] = compute_bio_scores(gut)
    
    for i in range(n):
        gut = (1 - alpha) * gut + alpha * signals[i]
        trajectory[i+1] = gut
        bio_trajectory[i+1] = compute_bio_scores(gut)
    
    return trajectory, bio_trajectory

traj_ema, bio_ema = simulate_ema(food_signals, alpha=0.08)
print('EMA simulation complete')
print(f'Health range: [{bio_ema[:,0].min():.3f}, {bio_ema[:,0].max():.3f}]')
print(f'Health std: {bio_ema[100:,0].std():.4f} (after warmup)')

EMA simulation complete
Health range: [-0.913, 2.543]
Health std: 0.5563 (after warmup)


---
## Model 2: Dual Fast/Slow Layers

- **Slow layer** (α=0.02): deep accumulation, drives the continuous AV state
- **Fast layer** (α=0.35): responsive to recent selections
- **Impulse** = fast - slow: captures the "disruption" from each selection

In [4]:
def simulate_dual_layer(signals, alpha_slow=0.02, alpha_fast=0.35):
    n, d = signals.shape
    gut_slow = np.zeros(d)
    gut_fast = np.zeros(d)
    
    traj_slow = np.zeros((n+1, d))
    traj_fast = np.zeros((n+1, d))
    bio_slow = np.zeros((n+1, len(BIO_NAMES)))
    bio_fast = np.zeros((n+1, len(BIO_NAMES)))
    bio_impulse = np.zeros((n+1, len(BIO_NAMES)))
    
    bio_slow[0] = compute_bio_scores(gut_slow)
    bio_fast[0] = compute_bio_scores(gut_fast)
    
    for i in range(n):
        gut_slow = (1 - alpha_slow) * gut_slow + alpha_slow * signals[i]
        gut_fast = (1 - alpha_fast) * gut_fast + alpha_fast * signals[i]
        
        traj_slow[i+1] = gut_slow
        traj_fast[i+1] = gut_fast
        bio_slow[i+1] = compute_bio_scores(gut_slow)
        bio_fast[i+1] = compute_bio_scores(gut_fast)
        bio_impulse[i+1] = bio_fast[i+1] - bio_slow[i+1]
    
    return traj_slow, traj_fast, bio_slow, bio_fast, bio_impulse

traj_slow, traj_fast, bio_slow, bio_fast, bio_impulse = simulate_dual_layer(food_signals)
print('Dual-layer simulation complete')
print(f'Slow health range: [{bio_slow[:,0].min():.3f}, {bio_slow[:,0].max():.3f}], std={bio_slow[100:,0].std():.4f}')
print(f'Fast health range: [{bio_fast[:,0].min():.3f}, {bio_fast[:,0].max():.3f}], std={bio_fast[100:,0].std():.4f}')
print(f'Impulse health range: [{bio_impulse[:,0].min():.3f}, {bio_impulse[:,0].max():.3f}], std={bio_impulse[100:,0].std():.4f}')

Dual-layer simulation complete
Slow health range: [0.000, 1.709], std=0.3099
Fast health range: [-2.135, 3.928], std=1.0899
Impulse health range: [-2.798, 2.673], std=0.9968


---
## Model 3: Physics / Momentum Model

The gut state is a particle with position and velocity. Each food applies a force. State-dependent sensitivity modulates force based on how "surprising" the food is.

- `force = (food_signal - position) * sensitivity_scale`
- `sensitivity_scale = 1 + sensitivity * distance(food_signal, position)`
- `velocity = velocity * drag + force * force_scale`
- `position = position + velocity`

In [5]:
def simulate_physics(signals, drag=0.85, force_scale=0.05, sensitivity=2.0):
    n, d = signals.shape
    pos = np.zeros(d)
    vel = np.zeros(d)
    
    traj_pos = np.zeros((n+1, d))
    traj_vel = np.zeros((n+1, d))
    bio_pos = np.zeros((n+1, len(BIO_NAMES)))
    bio_vel_mag = np.zeros(n+1)
    
    bio_pos[0] = compute_bio_scores(pos)
    
    for i in range(n):
        diff = signals[i] - pos
        dist = np.linalg.norm(diff)
        sens = 1.0 + sensitivity * dist
        force = diff * force_scale * sens
        
        vel = vel * drag + force
        pos = pos + vel
        
        traj_pos[i+1] = pos
        traj_vel[i+1] = vel
        bio_pos[i+1] = compute_bio_scores(pos)
        bio_vel_mag[i+1] = np.linalg.norm(compute_bio_scores(vel))
    
    return traj_pos, traj_vel, bio_pos, bio_vel_mag

traj_phys, traj_vel, bio_phys, bio_vel_mag = simulate_physics(food_signals)
print('Physics simulation complete')
print(f'Health range: [{bio_phys[:,0].min():.3f}, {bio_phys[:,0].max():.3f}], std={bio_phys[100:,0].std():.4f}')
print(f'Velocity magnitude range: [{bio_vel_mag[1:].min():.4f}, {bio_vel_mag[1:].max():.4f}]')

Physics simulation complete
Health range: [-6.066, 7.249], std=1.9399
Velocity magnitude range: [0.0061, 3.1972]


---
## Comparison: Health Score Trajectories

Side-by-side comparison of how the health bio-score evolves under each model.

In [6]:
fig = make_subplots(rows=4, cols=1, subplot_titles=[
    'Model 1: Original EMA (α=0.08)',
    'Model 2: Dual Layer — Slow (α=0.02)',
    'Model 2: Dual Layer — Fast (α=0.35)',
    'Model 3: Physics (drag=0.85, force=0.05, sensitivity=2.0)',
], vertical_spacing=0.06, shared_xaxes=True)

fig.add_trace(go.Scatter(y=bio_ema[:, 0], mode='lines', line=dict(width=1, color='#636EFA'),
    name='EMA'), row=1, col=1)
fig.add_trace(go.Scatter(y=bio_slow[:, 0], mode='lines', line=dict(width=1, color='#EF553B'),
    name='Slow'), row=2, col=1)
fig.add_trace(go.Scatter(y=bio_fast[:, 0], mode='lines', line=dict(width=1.5, color='#00CC96'),
    name='Fast'), row=3, col=1)
fig.add_trace(go.Scatter(y=bio_phys[:, 0], mode='lines', line=dict(width=1, color='#AB63FA'),
    name='Physics'), row=4, col=1)

for i in range(1, 5):
    fig.update_yaxes(title_text='Health', row=i, col=1)
fig.update_xaxes(title_text='Food Selection #', row=4, col=1)
fig.update_layout(template=PLOTLY_DARK, height=900, width=1100,
    title='Health Score Evolution: 2,000 Food Selections', showlegend=False)
fig.show()

---
## All Bio-Scores: Model Comparison

How much variation does each model produce across all 8 bio-score dimensions?

In [7]:
fig = make_subplots(rows=len(BIO_NAMES), cols=3,
    subplot_titles=[f'{name} — {model}' for name in BIO_NAMES for model in ['EMA', 'Dual Slow', 'Physics']],
    vertical_spacing=0.03, horizontal_spacing=0.05)

for i, name in enumerate(BIO_NAMES):
    row = i + 1
    warmup = 50
    fig.add_trace(go.Scatter(y=bio_ema[warmup:, i], mode='lines', line=dict(width=0.8, color='#636EFA'),
        showlegend=False), row=row, col=1)
    fig.add_trace(go.Scatter(y=bio_slow[warmup:, i], mode='lines', line=dict(width=0.8, color='#EF553B'),
        showlegend=False), row=row, col=2)
    fig.add_trace(go.Scatter(y=bio_phys[warmup:, i], mode='lines', line=dict(width=0.8, color='#AB63FA'),
        showlegend=False), row=row, col=3)

fig.update_layout(template=PLOTLY_DARK, height=200*len(BIO_NAMES), width=1200,
    title='All Bio-Scores: EMA vs Dual-Slow vs Physics (after 50-step warmup)')
fig.show()

---
## Variability Statistics

Quantify how much "movement" each model produces. More variability = more diverse AV states.

In [8]:
warmup = 100

stats = pd.DataFrame(index=BIO_NAMES)
for i, name in enumerate(BIO_NAMES):
    stats.loc[name, 'EMA_std'] = bio_ema[warmup:, i].std()
    stats.loc[name, 'EMA_range'] = bio_ema[warmup:, i].max() - bio_ema[warmup:, i].min()
    stats.loc[name, 'Dual_slow_std'] = bio_slow[warmup:, i].std()
    stats.loc[name, 'Dual_slow_range'] = bio_slow[warmup:, i].max() - bio_slow[warmup:, i].min()
    stats.loc[name, 'Dual_fast_std'] = bio_fast[warmup:, i].std()
    stats.loc[name, 'Dual_fast_range'] = bio_fast[warmup:, i].max() - bio_fast[warmup:, i].min()
    stats.loc[name, 'Physics_std'] = bio_phys[warmup:, i].std()
    stats.loc[name, 'Physics_range'] = bio_phys[warmup:, i].max() - bio_phys[warmup:, i].min()
    stats.loc[name, 'Impulse_std'] = bio_impulse[warmup:, i].std()
    stats.loc[name, 'Impulse_range'] = bio_impulse[warmup:, i].max() - bio_impulse[warmup:, i].min()

print('Variability Statistics (after 100-step warmup):')
print(stats.round(5).to_string())

print('\n\nVariability ratios (relative to EMA baseline):')
for model in ['Dual_slow', 'Dual_fast', 'Physics', 'Impulse']:
    ratio = stats[f'{model}_std'] / stats['EMA_std']
    print(f'\n  {model} / EMA (std ratio):')
    for name in BIO_NAMES:
        r = ratio[name]
        bar = '█' * int(min(r * 10, 50))
        print(f'    {name:15s}: {r:.2f}x  {bar}')

Variability Statistics (after 100-step warmup):
              EMA_std  EMA_range  Dual_slow_std  Dual_slow_range  Dual_fast_std  Dual_fast_range  Physics_std  Physics_range  Impulse_std  Impulse_range
health        0.55632    3.45579        0.30989          1.60869        1.08988          6.06281      1.93987       13.31440      0.99678        5.47076
butyrate      0.00826    0.05127        0.00436          0.02305        0.01619          0.09821      0.02818        0.20855      0.01486        0.08864
GABA          0.00556    0.02937        0.00277          0.01481        0.01097          0.05824      0.01667        0.10894      0.01012        0.05550
serotonin     0.00682    0.04091        0.00389          0.01879        0.01300          0.07248      0.02193        0.15327      0.01184        0.06652
dopamine      0.01225    0.07415        0.00658          0.03306        0.02444          0.15269      0.04399        0.33249      0.02244        0.13977
inflammation  0.01229    0.07423  

---
## Impulse Layer: Event Detection

The impulse (fast - slow) captures moments of disruption. When does it spike? What foods cause the biggest impulses?

In [9]:
impulse_magnitude = np.linalg.norm(bio_impulse, axis=1)

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Impulse Magnitude Over Time',
    'Top 20 Largest Impulses (food disruption events)',
], vertical_spacing=0.12)

fig.add_trace(go.Scatter(y=impulse_magnitude, mode='lines',
    line=dict(width=1, color='orange'), showlegend=False), row=1, col=1)

top20_idx = np.argsort(impulse_magnitude)[-20:][::-1]
fig.add_trace(go.Bar(
    x=[f'#{i}: {sequence[i-1]}' if i > 0 else '#0' for i in top20_idx],
    y=[impulse_magnitude[i] for i in top20_idx],
    marker_color='orange', showlegend=False), row=2, col=1)

fig.update_layout(template=PLOTLY_DARK, height=600, width=1100,
    title='Impulse Layer: Disruption Events')
fig.update_xaxes(tickangle=-45, row=2, col=1)
fig.show()

print('Top 10 impulse events:')
for idx in top20_idx[:10]:
    food = sequence[idx-1] if idx > 0 else '(start)'
    print(f'  Step {idx}: {food:20s}  magnitude={impulse_magnitude[idx]:.4f}  health_impulse={bio_impulse[idx,0]:.4f}')

Top 10 impulse events:
  Step 1862: bone_broth            magnitude=2.7997  health_impulse=-2.7980
  Step 1861: bone_broth            magnitude=2.7882  health_impulse=-2.7865
  Step 1860: bone_broth            magnitude=2.7396  health_impulse=-2.7379
  Step 478: bone_broth            magnitude=2.7387  health_impulse=-2.7370
  Step 1632: energy_drink          magnitude=2.7357  health_impulse=-2.7341
  Step 477: bone_broth            magnitude=2.7151  health_impulse=-2.7134
  Step 857: soda                  magnitude=2.7040  health_impulse=-2.7023
  Step 1980: bone_broth            magnitude=2.6918  health_impulse=-2.6902
  Step 1979: bone_broth            magnitude=2.6864  health_impulse=-2.6848
  Step 984: chia_seeds            magnitude=2.6752  health_impulse=2.6728


---
## State-Dependent Sensitivity: Does the Current State Affect Response?

In the physics model, foods that are "surprising" (far from current state) have larger effects. Visualize this.

In [10]:
distances = np.linalg.norm(food_signals - traj_phys[:-1], axis=1)
step_changes = np.linalg.norm(np.diff(traj_phys, axis=0), axis=1)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Food-State Distance vs Step Size (Physics Model)',
    'Distribution of Step Sizes'
])

fig.add_trace(go.Scatter(x=distances, y=step_changes, mode='markers',
    marker=dict(size=2, color=bio_phys[1:, 0], colorscale='RdYlGn', showscale=True,
                colorbar=dict(title='Health', x=0.45)),
    showlegend=False), row=1, col=1)

fig.add_trace(go.Histogram(x=step_changes, nbinsx=60, marker_color='#AB63FA',
    showlegend=False), row=1, col=2)

fig.update_xaxes(title_text='Distance (food vs current state)', row=1, col=1)
fig.update_yaxes(title_text='Step size', row=1, col=1)
fig.update_xaxes(title_text='Step size', row=1, col=2)
fig.update_layout(template=PLOTLY_DARK, height=450, width=1100,
    title='State-Dependent Sensitivity: Surprising Foods → Bigger Steps')
fig.show()

corr = np.corrcoef(distances, step_changes)[0, 1]
print(f'Correlation between food-state distance and step size: {corr:.3f}')

Correlation between food-state distance and step size: 0.324


---
## Combined Model: Dual Layer + State-Dependent Sensitivity

Best of both: slow/fast layers for temporal structure, plus sensitivity modulation for responsiveness.

In [11]:
def simulate_combined(signals, alpha_slow=0.02, alpha_fast=0.35,
                      sensitivity=1.5, fast_sensitivity=True):
    n, d = signals.shape
    gut_slow = np.zeros(d)
    gut_fast = np.zeros(d)
    
    bio_slow_out = np.zeros((n+1, len(BIO_NAMES)))
    bio_fast_out = np.zeros((n+1, len(BIO_NAMES)))
    bio_impulse_out = np.zeros((n+1, len(BIO_NAMES)))
    effective_alphas = np.zeros(n)
    
    for i in range(n):
        diff_slow = signals[i] - gut_slow
        dist_slow = np.linalg.norm(diff_slow)
        sens_factor = 1.0 + sensitivity * dist_slow
        
        eff_alpha_slow = min(alpha_slow * sens_factor, 0.15)
        eff_alpha_fast = min(alpha_fast * (sens_factor if fast_sensitivity else 1.0), 0.8)
        effective_alphas[i] = eff_alpha_slow
        
        gut_slow = (1 - eff_alpha_slow) * gut_slow + eff_alpha_slow * signals[i]
        gut_fast = (1 - eff_alpha_fast) * gut_fast + eff_alpha_fast * signals[i]
        
        bio_slow_out[i+1] = compute_bio_scores(gut_slow)
        bio_fast_out[i+1] = compute_bio_scores(gut_fast)
        bio_impulse_out[i+1] = bio_fast_out[i+1] - bio_slow_out[i+1]
    
    return bio_slow_out, bio_fast_out, bio_impulse_out, effective_alphas

bio_comb_slow, bio_comb_fast, bio_comb_impulse, eff_alphas = simulate_combined(food_signals)
print('Combined model simulation complete')
print(f'Slow health: range=[{bio_comb_slow[100:,0].min():.3f}, {bio_comb_slow[100:,0].max():.3f}], std={bio_comb_slow[100:,0].std():.4f}')
print(f'Effective alpha_slow: mean={eff_alphas.mean():.4f}, range=[{eff_alphas.min():.4f}, {eff_alphas.max():.4f}]')

Combined model simulation complete
Slow health: range=[-0.302, 1.988], std=0.3927
Effective alpha_slow: mean=0.0339, range=[0.0240, 0.0496]


In [12]:
fig = make_subplots(rows=4, cols=1, subplot_titles=[
    'Combined Model: Slow Layer (continuous AV state)',
    'Combined Model: Fast Layer (responsive)',
    'Combined Model: Impulse (fast - slow, event trigger)',
    'Effective Alpha (slow) — Higher = more surprising food',
], vertical_spacing=0.07, shared_xaxes=True)

fig.add_trace(go.Scatter(y=bio_comb_slow[:, 0], mode='lines',
    line=dict(width=1.5, color='#EF553B'), name='Slow health'), row=1, col=1)
fig.add_trace(go.Scatter(y=bio_comb_fast[:, 0], mode='lines',
    line=dict(width=1, color='#00CC96'), name='Fast health'), row=2, col=1)
fig.add_trace(go.Scatter(y=bio_comb_impulse[:, 0], mode='lines',
    line=dict(width=1, color='orange'), name='Impulse health'), row=3, col=1)
fig.add_trace(go.Scatter(y=eff_alphas, mode='lines',
    line=dict(width=1, color='white'), name='Eff. alpha'), row=4, col=1)

fig.update_yaxes(title_text='Health', row=1, col=1)
fig.update_yaxes(title_text='Health', row=2, col=1)
fig.update_yaxes(title_text='Impulse', row=3, col=1)
fig.update_yaxes(title_text='Alpha', row=4, col=1)
fig.update_xaxes(title_text='Food Selection #', row=4, col=1)
fig.update_layout(template=PLOTLY_DARK, height=900, width=1100,
    title='Combined Model: Dual Layer + State-Dependent Sensitivity', showlegend=False)
fig.show()

---
## Final Comparison: Variability Across All Models

In [13]:
warmup = 100

models = {
    'EMA (α=0.08)': bio_ema,
    'Dual: Slow (α=0.02)': bio_slow,
    'Dual: Fast (α=0.35)': bio_fast,
    'Dual: Impulse': bio_impulse,
    'Physics': bio_phys,
    'Combined: Slow': bio_comb_slow,
    'Combined: Fast': bio_comb_fast,
    'Combined: Impulse': bio_comb_impulse,
}

summary = pd.DataFrame()
for model_name, bio_data in models.items():
    for i, dim in enumerate(BIO_NAMES):
        data = bio_data[warmup:, i]
        summary.loc[model_name, f'{dim}_std'] = data.std()
        summary.loc[model_name, f'{dim}_range'] = data.max() - data.min()
        summary.loc[model_name, f'{dim}_iqr'] = np.percentile(data, 75) - np.percentile(data, 25)

std_cols = [c for c in summary.columns if c.endswith('_std')]
std_df = summary[std_cols].copy()
std_df.columns = [c.replace('_std', '') for c in std_df.columns]

fig = px.imshow(std_df.values, x=list(std_df.columns), y=list(std_df.index),
    color_continuous_scale='Viridis', aspect='auto',
    title='Standard Deviation of Bio-Scores by Model (after warmup)',
    template=PLOTLY_DARK, width=900, height=450)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print('Standard deviation comparison (higher = more AV variety):')
print(std_df.round(5).to_string())

Standard deviation comparison (higher = more AV variety):
                      health  butyrate     GABA  serotonin  dopamine  inflammation  barrier  diversity
EMA (α=0.08)         0.55632   0.00826  0.00556    0.00682   0.01225       0.01229  0.00560    0.00243
Dual: Slow (α=0.02)  0.30989   0.00436  0.00277    0.00389   0.00658       0.00683  0.00288    0.00144
Dual: Fast (α=0.35)  1.08988   0.01619  0.01097    0.01300   0.02444       0.02299  0.01098    0.00490
Dual: Impulse        0.99678   0.01486  0.01012    0.01184   0.02244       0.02091  0.01009    0.00478
Physics              1.93987   0.02818  0.01667    0.02193   0.04399       0.03995  0.01684    0.00911
Combined: Slow       0.39268   0.00575  0.00359    0.00481   0.00871       0.00901  0.00369    0.00181
Combined: Fast       1.47437   0.02204  0.01439    0.01693   0.03374       0.03155  0.01457    0.00663
Combined: Impulse    1.36515   0.02044  0.01336    0.01560   0.03132       0.02904  0.01350    0.00658


In [14]:
print('\n' + '='*80)
print('DYNAMICS SIMULATION SUMMARY')
print('='*80)

print('\n1. ORIGINAL EMA (α=0.08)')
print(f'   Health std: {bio_ema[warmup:,0].std():.4f}')
print(f'   Problem: converges to center, low responsiveness to individual selections')

print('\n2. DUAL LAYER')
print(f'   Slow std:    {bio_slow[warmup:,0].std():.4f} (even smoother than EMA — slow accumulation)')
print(f'   Fast std:    {bio_fast[warmup:,0].std():.4f} (much more responsive)')
print(f'   Impulse std: {bio_impulse[warmup:,0].std():.4f} (captures disruption events)')
print(f'   Benefit: fast layer makes individual selections visible, slow layer provides tonal depth')

print('\n3. PHYSICS MODEL')
print(f'   Health std: {bio_phys[warmup:,0].std():.4f}')
print(f'   Benefit: state-dependent sensitivity makes "surprising" foods more impactful')

print('\n4. COMBINED MODEL (recommended)')
print(f'   Slow std:    {bio_comb_slow[warmup:,0].std():.4f}')
print(f'   Fast std:    {bio_comb_fast[warmup:,0].std():.4f}')
print(f'   Impulse std: {bio_comb_impulse[warmup:,0].std():.4f}')
print(f'   Eff alpha range: [{eff_alphas.min():.4f}, {eff_alphas.max():.4f}]')
print(f'   Benefit: combines temporal layering with sensitivity modulation')

print('\n5. AV MAPPING SIGNALS (from combined model):')
print(f'   → Slow bio-scores (8D): drives continuous AV texture (synth params, LED colors)')
print(f'   → Impulse bio-scores (8D): drives transient events (grains, flashes, formant shifts)')
print(f'   → Impulse magnitude (1D): drives event intensity')
print(f'   → Effective alpha (1D): drives "surprise" parameter')
print(f'   Total: up to 18 continuous control signals')


DYNAMICS SIMULATION SUMMARY

1. ORIGINAL EMA (α=0.08)
   Health std: 0.5563
   Problem: converges to center, low responsiveness to individual selections

2. DUAL LAYER
   Slow std:    0.3099 (even smoother than EMA — slow accumulation)
   Fast std:    1.0899 (much more responsive)
   Impulse std: 0.9968 (captures disruption events)
   Benefit: fast layer makes individual selections visible, slow layer provides tonal depth

3. PHYSICS MODEL
   Health std: 1.9399
   Benefit: state-dependent sensitivity makes "surprising" foods more impactful

4. COMBINED MODEL (recommended)
   Slow std:    0.3927
   Fast std:    1.4744
   Impulse std: 1.3652
   Eff alpha range: [0.0240, 0.0496]
   Benefit: combines temporal layering with sensitivity modulation

5. AV MAPPING SIGNALS (from combined model):
   → Slow bio-scores (8D): drives continuous AV texture (synth params, LED colors)
   → Impulse bio-scores (8D): drives transient events (grains, flashes, formant shifts)
   → Impulse magnitude (1D): d